# GPU check of the CPT pipeline on a free T4 (Colab / Kaggle)

Tests the parts of `scripts/train` that the CPU dry run cannot, before paying for the g6e
(docs/prepare_cpt_training_pipeline.md, sections 3.4, 4.4 and 9):

| Check | Cell |
|---|---|
| The packages install: `transformers==5.19.0`, `torchao`, `flash-linear-attention`, `causal-conv1d` | 3 |
| The DeltaNet kernels give the same outputs and gradients as transformers' PyTorch reference | 4 |
| 8-bit AdamW with bf16 stochastic rounding really moves bf16 weights | 4 |
| The whole pipeline on a GPU with the real optimizer and kernels: train, eval, probes, gates, epoch copies, stop + resume | 5–7 |

**Not testable here:** whether the real 4B model fits in 48 GB, and its speed. That is the g6e smoke test (~10 min).

**Before you start:** *Runtime → Change runtime type → T4 GPU* (Kaggle: *Settings → Accelerator → GPU T4 x2*, and
*Internet on*). Then *Runtime → Run all*, or run the cells one by one. Total ~15–40 min, mostly installing.

A T4 has no native bf16 (the L40S has). If only the **bfloat16** rows fail in cell 4, that is the T4, not the setup:
set `USE_BF16 = False` in cell 5 and run on.

In [ ]:
# 1. GPU visible? (Kaggle's "T4 x2" shows 2 GPUs: training uses one, as on the g6e)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["WANDB_MODE"] = "disabled"
!nvidia-smi --query-gpu=name,memory.total,driver_version,compute_cap --format=csv

In [ ]:
# 2. Get the code and the packed dataset (the repo is public)
!git clone --depth 1 https://github.com/kanishkapg/Sinhala-LLM.git /content/Sinhala-LLM 2>&1 | tail -1
%cd /content/Sinhala-LLM
!git log --oneline -1

In [ ]:
# 3. Install what setup_machine.sh installs on the g6e (keeps the platform's CUDA torch). causal-conv1d may compile
#    for 5-20 min if no prebuilt wheel matches this torch: let it run.
!pip install -q -r scripts/train/requirements-train.txt
!pip install -q causal-conv1d --no-build-isolation
!pip list 2>/dev/null | grep -iE "^(torch|torchao|transformers|flash-linear-attention|causal-conv1d|triton) "

In [ ]:
# 4. Kernels vs PyTorch reference + optimizer rounding. Read the summary at the end.
!python scripts/train/gpu_check.py

In [ ]:
# 5. Settings for the pipeline runs. USE_BF16 = False if only the bfloat16 rows failed in cell 4 (T4 limit).
USE_BF16 = True

!python scripts/train/make_tiny_model.py --real-heads --out outputs/tiny-qwen3.5-gpu 2>&1 | tail -1
DTYPE = "" if USE_BF16 else "model_dtype=float32 bf16=false"
# the tiny random model with the 4B head sizes, the REAL optimizer of cpt_easy.json, and the kernels required
SET = (f"--set model=outputs/tiny-qwen3.5-gpu optim=adamw_torch_8bit optim_args=bf16_stochastic_round=True "
       f"allow_slow_kernels=false {DTYPE}")
CFG = "--config scripts/train/config/debug_tiny.json"
print(SET)

In [ ]:
# 6. Smoke mode on the GPU. Expected: "first training loss" FAIL (random model, ~12.4), every other line PASS,
#    in particular "DeltaNet kernels, no fallback warning".
!python scripts/train/train_cpt.py {CFG} --smoke {SET} 2>&1 | grep -vE "it/s\]|s/it\]" | tail -25

In [ ]:
# 7. Stop at step 6, resume to the end (12 steps, epoch copies at 4 / 8 / 12)
!rm -rf outputs/debug-gpu
!python scripts/train/train_cpt.py {CFG} --stop-at-step 6 {SET} output_dir=outputs/debug-gpu 2>&1 | grep -E "probes step|Check|epoch copy|stopped|Error|Traceback|WARNING"
!python scripts/train/train_cpt.py {CFG} --resume {SET} output_dir=outputs/debug-gpu 2>&1 | grep -E "probes step|Check|epoch copy|finished|Error|Traceback|WARNING"

In [ ]:
# 8. Verify the run
import json, math, glob, torch
last = max(glob.glob("outputs/debug-gpu/checkpoint-*"), key=lambda p: int(p.split("-")[-1]))
state = json.load(open(last + "/trainer_state.json"))
hist = state["log_history"]
losses = {h["step"]: h["loss"] for h in hist if "loss" in h}
targs = torch.load(last + "/training_args.bin", weights_only=False)
checks = {
    "12 steps done": state["global_step"] == 12,
    "loss logged at every step": sorted(losses) == list(range(1, 13)),
    "no NaN / inf loss": all(math.isfinite(v) for v in losses.values()),
    "loss went down (random model, lr 1e-3)": losses[12] < losses[1],
    # 0/4/8/12 from eval_steps=4, plus step 6 where --stop-at-step evaluates before stopping
    "validation losses at 0/4/6/8/12": {0, 4, 6, 8, 12} <= {h["step"] for h in hist if "eval_textbook_loss" in h},
    "probes at 0/4/6/8/12": [h["step"] for h in hist if "eval_probe_a_acc" in h] == [0, 4, 6, 8, 12],
    "Check 1 and Check 2 ran once": sorted(k for h in hist for k in h if "gate" in k) == ["eval_gate_check1", "eval_gate_check2"],
    "epoch copies e1 e2 e3": all(glob.glob(f"outputs/debug-gpu/epochs/{e}/model.safetensors") for e in ("e1", "e2", "e3")),
    "optimizer = 8-bit AdamW + stochastic rounding": getattr(targs.optim, "value", targs.optim) == "adamw_torch_8bit" and "bf16_stochastic_round=True" in str(targs.optim_args),
}
for k, ok in checks.items():
    print(("PASS " if ok else "FAIL ") + k)
print("loss:", {s: round(v, 3) for s, v in sorted(losses.items())})
print()
print("ALL PASSED" if all(checks.values()) else "SOMETHING FAILED: send the output of cells 4-8")

In [ ]:
# 9. Reload test: load the saved epoch copy on the GPU and score it with the probe code
!python scripts/train/probe_eval.py --model outputs/debug-gpu/epochs/e3 --out outputs/probes/debug-gpu --limit 50 --max-new-tokens 16 2>&1 | grep -E "parameters|probe_|Error|Traceback" | head -8

## Reading the results

| Where | Good | Means |
|---|---|---|
| Cell 3 | the `pip list` shows all 6 packages | the installs work with a recent CUDA torch |
| Cell 4 summary | `RESULT: all needed checks passed` (bfloat16 rows may fail / WARN on a T4) | kernels compute the same as the reference; stochastic rounding works |
| Cell 6 | every line PASS except "first training loss" | the training loop uses the fast kernels, no fallback |
| Cell 8 | `ALL PASSED` | the full pipeline runs on a GPU with the real optimizer |
| Cell 9 | `embeddings tied: True` and probe lines | saved models reload |

A failure: copy the output of cells 3–9 into the chat. The g6e's AMI has a different torch / CUDA than Colab, so
`setup_machine.sh` and `gpu_check.py` run again there before the smoke test.